# Antarctic Meltwater Detection from Sentinel-1 NRB
This notebook follows the methods on Liang et al. (2021) with minor adaptions. To detect meltwater in Sentinel-1 Images:
- Identify the winter baseline for each burst using the geomedian of HH NRB between June 1 - June 31 for the previous winter season.
- Subtract the winter baseline from the burst 
- Use a threshold of -2.66 dB to determine meltwater area

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from pystac_client import Client
from odc.stac import load, configure_s3_access
from odc.geo import BoundingBox
from shapely.geometry import box
import geopandas as gpd
import numpy as np
import xarray as xr
from IPython.display import display
from pystac_client import Client
import numpy as np
import sys
sys.path.insert(1,'./src/')
from lee_sigma_improved_xr import lee_sigma_improved_xr, lee_sigma_improved_xr_multitime
from shapely.geometry import box
import geopandas as gpd
import pandas as pd

#### Robinson Ridge Area Of Ineterest (AOI)
We are interested in detecting the meltwater conditions of robinson ridge. As the provided AOI is small and only covers a space with ~10 pixels, I expand the AOI to the entire peninsular.

In [ ]:
# Convert the small bounding box to EPSG 3031 for later plotting
melt_bbox = BoundingBox(110.575,-66.365, 110.620,-66.378,crs="EPSG:4326") # Larger Robinson Ridge AOI
rr_AOI = BoundingBox(110.5847239139500005,-66.3689977259193000, 110.5866999110109958,-66.3676057235114030,crs="EPSG:4326") # Robinson Ridge AOI

In [ ]:
# Convert BoundingBox → shapely polygon
bbox_poly = box(110.5847239139500005,-66.3689977259193000, 110.5866999110109958,-66.3676057235114030)

# Put into GeoDataFrame with WGS84
gdf = gpd.GeoDataFrame(geometry=[bbox_poly], crs="EPSG:4326")

# Reproject to EPSG:3031
gdf_3031 = gdf.to_crs("EPSG:3031")
bb_x, bb_y = gdf_3031.geometry[0].exterior.xy

### Access Sentinel-1
Query DE development STAC API (unstable)

*Note this API is unstable and could be periodically shut down

In [ ]:
# Set up the query
catalog = "https://explorer.dev.dea.ga.gov.au/stac"
stac_client = Client.open(catalog)
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

In [ ]:
# retrieve items from the STAC API
melt_items = stac_client.search(
    #datetime=f"{start}/{end}",
    collections=['ga_s1_nrb_iw_hh_0'],
    bbox = melt_bbox.bbox
).item_collection()
print(f"Found {len(melt_items)} items")

In [ ]:
# Assets to load
assets_to_load = ["HH_gamma0","number_of_looks"]

# CRS and resolution
output_crs = "epsg:3031"
output_res = 20 # 20m is the native resolution of the data.

# Property or function to group by. "solar_day" is already built into odc-stac
groupy_by_operation = "solar_day"

In [ ]:
# Load the data
ds = load(
    items=melt_items,
    bands=assets_to_load,
    intersects = melt_bbox.boundary(),
    crs=output_crs,
    resolution=output_res,
    groupby=groupy_by_operation,
    chunks={},
)


In [ ]:
# See what we have
ds

In [ ]:
ds.x.values.min(), ds.x.values.max(), ds.y.values.min(), ds.y.values.max()


In [ ]:
# convert to dB
ds['HH_gamma0_dB'] = 10 * np.log10(ds['HH_gamma0'])
# plot the results to check
ds['HH_gamma0_dB']\
    .isel(time=slice(0, 10))\
    .plot.imshow(col="time", col_wrap=2)

### Speckle filtering using the lee improved filter with adaptive multilooking 

Apply the lee improves speckle filter using the number of looks information provided by ISCE3

In [ ]:
# Apply speckle filter to the data using the number of looks information provided by ISCE3
filtered_ds = lee_sigma_improved_xr_multitime(
    ds,
    band_name="HH_gamma0",
    nlooks="number_of_looks",
    win=(9, 9),
    sigma=0.9,
    perc=0.02,
    data_type="intensity",
    verbose=True,
)

Convert to decibels for visualisation

In [ ]:
# convert to dB
filtered_ds['HH_gamma0_lee_sigma_improved_dB'] = 10 * np.log10(filtered_ds['HH_gamma0_lee_sigma_improved'])

In [ ]:
# plot the results to check
filtered_ds['HH_gamma0_lee_sigma_improved_dB']\
    .isel(time=slice(0, 10))\
    .plot.imshow(col="time", col_wrap=2)

In [ ]:
# check the data
filtered_ds

### Normalise the speckle filtered data
Data are normalised to the previous winter year. This is calculated using the median of NRB observed between June 1 and July 31st. Median values for each pixel are subtracted from the speckle filtered data.

Calculate the median for each winter year

In [ ]:
import xarray as xr
import pandas as pd

# --- User inputs ---
input_ds = filtered_ds  # your xarray.Dataset
data_var = "HH_gamma0_lee_sigma_improved_dB"  # variable to compute median
# get first/last year from the time coordinate (works on the DataArray)
start_year = int(input_ds.time.dt.year.min().item())
end_year   = int(input_ds.time.dt.year.max().item())

# --- Prepare a list to hold yearly medians ---
median_list = []

for year in range(start_year, end_year + 1):
    # Select June and July data for this year
    ds_yr = input_ds.sel(
        time=slice(f"{year}-06-01", f"{year}-07-31")
    )
    
    if len(ds_yr.time) == 0:
        # Skip if no data
        continue
    
    # Compute median for the selected variable along 'time'
    median_da = ds_yr[data_var].median(dim="time", skipna=True)
    
    # Add year as a new coordinate
    median_da = median_da.expand_dims(time=[pd.Timestamp(f"{year}-07-01")])
    
    median_list.append(median_da)

# Combine all years into a single Dataset
median_ds = xr.concat(median_list, dim="time")
median_ds = median_ds.to_dataset(name=f"{data_var}_median")

# Optional: save to NetCDF
# median_ds.to_netcdf("yearly_june_july_median.nc")

print(median_ds)

Check medians

In [ ]:
# check the results
median_ds['HH_gamma0_lee_sigma_improved_dB_median'].plot.imshow(col="time", col_wrap=2)

Normalise the data and create a binary melt indicator

In [ ]:
# Copy the filtered_ds to avoid overwriting if you want
ds_norm = filtered_ds.copy()

# Create a Series mapping each timestamp to the correct median year
def get_median_year(ts):
    """Return the year of median to use for a given timestamp."""
    if ts.month >= 6:  # June to December -> same year
        return ts.year
    else:               # January to May -> previous year
        return ts.year - 1

# Apply to all timestamps
median_years = pd.Series(ds_norm.time.values).apply(get_median_year)

# Build an array of medians matching each time in filtered_ds
median_values = []
for ts, year in zip(ds_norm.time.values, median_years):
    # Find median for that year
    try:
        med_val = median_ds[f"{data_var}_median"].sel(time=f"{year}-07-01")
    except KeyError:
        # If no median for that year, fill with NaN
        med_val = xr.full_like(ds_norm[data_var].isel(time=0), fill_value=float('nan'))
    median_values.append(med_val)

# Stack medians along 'time' to match filtered_ds
median_aligned = xr.concat(median_values, dim="time")
median_aligned = median_aligned.assign_coords(time=ds_norm.time)

# Compute normalised variable
ds_norm[f"{data_var}_normalised"] = ds_norm[data_var] - median_aligned

# Optional: update filtered_ds in-place
filtered_ds[f"{data_var}_normalised"] = ds_norm[f"{data_var}_normalised"]

melt = (filtered_ds["HH_gamma0_lee_sigma_improved_dB_normalised"] < -2.66)

# create the binary melt variable, but only where the normalised variable is not NaN
filtered_ds["HH_gamma0_lee_sigma_improved_dB_melt"] = melt.where(
    filtered_ds["HH_gamma0_lee_sigma_improved_dB_normalised"].notnull()
)

print(filtered_ds)

In [ ]:
# check the normalised data
filtered_ds['HH_gamma0_lee_sigma_improved_dB_normalised']\
    .isel(time=slice(0, 10))\
    .plot.imshow(col="time", col_wrap=2)

### Mask the data for the coastline 

In [ ]:
# Load Antarctic coastline shapefile
antarctica = gpd.read_file("add_coastline_high_res_polygon_v7_4.shp").to_crs("EPSG:3031")  # ensure lon/lat

In [ ]:
# Get y/x grids
x = filtered_ds.x
y = filtered_ds.y

# Create 2D meshgrid of x/y
x2d, y2d = np.meshgrid(x, y)

# Fyten for easier point-in-polygon check
points = gpd.GeoSeries(gpd.points_from_xy(x2d.ravel(), y2d.ravel()), crs="EPSG:4326")

# Create mask: True if point is inside Antarctica
mask_fy = points.within(antarctica.unary_union)
mask = mask_fy.values.reshape(y.size, x.size)

# Convert to xarray DataArray aligned with filtered_ds
mask_da = xr.DataArray(mask, coords={"y": y, "x": x}, dims=["y", "x"])

In [ ]:
# Use where() to mask values outside Antarctica
filtered_ds_masked = filtered_ds.where(mask_da)

# Optionally, also mask the normalized variable
# filtered_ds_masked[f"{data_var}_normalized"] = filtered_ds[f"{data_var}_normalized"].where(mask_da)

In [ ]:
#check the masked melt variable
filtered_ds_masked['HH_gamma0_lee_sigma_improved_dB_melt']\
    .isel(time=slice(0, 10))\
    .plot.imshow(col="time", col_wrap=2)

### Visualisation through time

In [ ]:
# ...existing code...
import numpy as np

# Use the normalised variable for animation
da = filtered_ds_masked['HH_gamma0_lee_sigma_improved_dB']

# Buffer settings: choose either buffer_m (meters) or buffer_pixels (pixels)
# Set buffer_m = 20 for a 20 m buffer, or set buffer_pixels = 1 to buffer by one pixel.
buffer_m = 20
buffer_pixels = None  # set to integer (e.g. 1) to use pixel buffer instead

# If using pixel buffer convert to meters using output_res (if available)
if buffer_pixels is not None:
    try:
        pixel_size = output_res  # defined earlier in the notebook
    except NameError:
        pixel_size = 20  # fallback if output_res not defined
    buffer_m = buffer_pixels * pixel_size

# Create buffered polygon in projected CRS (gdf_3031 is EPSG:3031)
buffered_poly = gdf_3031.geometry[0].buffer(buffer_m)

# Save polygon coords for plotting (bb_x, bb_y used in animation cell)
bb_x, bb_y = buffered_poly.exterior.xy

# Get buffered bounds for slicing
minx, miny, maxx, maxy = buffered_poly.bounds

# Determine coordinate ordering and build safe slices
x_coords = filtered_ds_masked.x.values
y_coords = filtered_ds_masked.y.values

x_sel = slice(minx, maxx) if x_coords[0] < x_coords[-1] else slice(maxx, minx)
y_sel = slice(miny, maxy) if y_coords[0] < y_coords[-1] else slice(maxy, miny)

# Subset to buffered AOI
da = da.sel(x=x_sel, y=y_sel)

# Drop times that are entirely NaN (optional, keeps the animation tidy)
if 'time' in da.dims:
    da = da.dropna(dim='time', how='all')

da
# ...existing code...

In [ ]:
import matplotlib.animation as animation

fig, ax = plt.subplots()
# Initial plot
im = da.isel(time=0).plot.imshow(ax=ax, add_colorbar=True,vmin=-10, vmax=0 )

def update(frame):
    ax.clear()
    da.isel(time=frame).plot.imshow(
        ax=ax,
        add_colorbar=False,
        vmin=-10, vmax=0  # binary scale
    )
    ax.plot(bb_x, bb_y, color='cyan', linewidth=2)
    
    ax.set_title(str(da.time.values[frame]))


ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(da.time),
    interval=300  # milliseconds between frames
)

plt.show()
ani.save("antarctica_melt_animation_cropped.gif", writer="pillow", fps=3)

### Increase resolution using superresolution models
We will apply super-resolution models to the meltwater binary and normalised backscatter layers. We are doing this as a post-processing step to preserve the integrity of the normalisation process and to reduce error propegation. 

We are trying:
- Real-ESRGAN
- OpenCV

In [ ]:
#Real-ESRGAN super-resolution
import xarray as xr
import numpy as np
import torch
from realesrgan import RealESRGAN

# --- Select slice ---
da = filtered_ds['HH_gamma0_lee_sigma_improved_dB_normalized'].isel(time=0)

img = da.values

# --- Normalise to 0–255 ---
img_min, img_max = np.nanmin(img), np.nanmax(img)
img_norm = ((img - img_min) / (img_max - img_min) * 255).astype(np.uint8)

# Convert to 3-channel (Real-ESRGAN expects RGB)
img_rgb = np.stack([img_norm]*3, axis=-1)

# --- Load model ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = RealESRGAN(device, scale=4)
model.load_weights('RealESRGAN_x4.pth')  # auto-downloads if not present

# --- Apply SR ---
sr_img = model.predict(img_rgb)

# Convert back to single channel
sr_img = sr_img[..., 0]

# --- Back to float range ---
sr_img = sr_img.astype(np.float32) / 255.0
sr_img = sr_img * (img_max - img_min) + img_min

# --- Back to xarray ---
y = np.linspace(da.y.min(), da.y.max(), sr_img.shape[0])
x = np.linspace(da.x.min(), da.x.max(), sr_img.shape[1])

sr_da = xr.DataArray(
    sr_img,
    dims=("y", "x"),
    coords={"y": y, "x": x},
    name="SR_RealESRGAN"
)

In [ ]:
# ...existing code...
import cv2
import numpy as np
import xarray as xr

# Apply EDSR (OpenCV DNN) over every time slice of the masked dataset
src_name = "HH_gamma0_lee_sigma_improved_dB_normalised"
if "filtered_ds_masked" in globals() and src_name in filtered_ds_masked:
    da_all = filtered_ds_masked[src_name]
elif "filtered_ds" in globals() and src_name in filtered_ds:
    da_all = filtered_ds[src_name]
else:
    raise KeyError(f"{src_name} not found in filtered_ds_masked or filtered_ds")

# Load model once
sr = cv2.dnn_superres.DnnSuperResImpl_create()
sr.readModel("EDSR_x4.pb")
sr.setModel("edsr", 4)

sr_frames = []
for i, t in enumerate(da_all.time.values):
    da = da_all.isel(time=i)
    img = np.asarray(da.values, dtype=np.float32)

    # handle all-NaN frame
    if np.all(np.isnan(img)):
        # produce NaN upsampled array by using target shape from upsampling a dummy image
        dummy = np.zeros_like(img, dtype=np.uint8)
        up_dummy = sr.upsample(cv2.cvtColor(dummy, cv2.COLOR_GRAY2BGR)) if dummy.ndim == 2 else sr.upsample(dummy)
        up_shape = up_dummy.shape[:2]
        sr_float = np.full(up_shape, np.nan, dtype=np.float32)
    else:
        nan_mask = np.isnan(img)
        img_min = np.nanmin(img)
        img_max = np.nanmax(img)
        if img_max > img_min:
            img_norm = ((img - img_min) / (img_max - img_min) * 255.0).clip(0,255)
            img_norm = np.where(nan_mask, 0, img_norm).round().astype(np.uint8)
        else:
            # constant image -> create flat uint8 image
            img_norm = np.zeros_like(img, dtype=np.uint8)

        # ensure 3-channel BGR
        if img_norm.ndim == 2:
            img_in = cv2.cvtColor(img_norm, cv2.COLOR_GRAY2BGR)
        else:
            img_in = img_norm

        up = sr.upsample(img_in)  # returns HxWxC or HxW
        # take first channel if color
        if up.ndim == 3:
            up_chan = up[..., 0]
        else:
            up_chan = up

        # upsample original NaN mask to reapply as NaN (nearest)
        mask_up = cv2.resize((~nan_mask).astype(np.uint8), (up_chan.shape[1], up_chan.shape[0]), interpolation=cv2.INTER_NEAREST)
        valid_up = mask_up.astype(bool)

        sr_float = up_chan.astype(np.float32) / 255.0
        # map back to original float range only where original had valid pixels
        if img_max > img_min:
            sr_float = sr_float * (img_max - img_min) + img_min
        else:
            sr_float = sr_float * 0.0 + img_min

        # set invalid pixels to NaN
        sr_float[~valid_up] = np.nan

    # build coordinates for upsampled frame
    y0, y1 = float(da.y.min()), float(da.y.max())
    x0, x1 = float(da.x.min()), float(da.x.max())
    y_up = np.linspace(y0, y1, sr_float.shape[0])
    x_up = np.linspace(x0, x1, sr_float.shape[1])

    sr_da = xr.DataArray(sr_float, dims=("y", "x"), coords={"y": y_up, "x": x_up})
    sr_da = sr_da.expand_dims(time=[da.time.values])
    sr_frames.append(sr_da)

# concat into a single DataArray with time dimension
SR_OpenCV = xr.concat(sr_frames, dim="time")
SR_OpenCV.name = "SR_OpenCV"

SR_OpenCV
# ...existing code...

In [ ]:
SR_OpenCV 

In [ ]:
#check the masked melt variable
filtered_ds_masked['HH_gamma0_lee_sigma_improved_dB_melt']\
    .isel(time=slice(0, 10))\
    .plot.imshow(col="time", col_wrap=2)